# Current version : 9.A (2025-04-22)

# Libraries and directory (always run)

In [ ]:
### import necessary libraries
# import anndata as ad
from datetime import datetime
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import os
import pandas as pd
import random
import seaborn as sns
import scanpy as sc
import warnings

warnings.filterwarnings("ignore") 
sc.logging.print_header()
sc.set_figure_params(facecolor="white", figsize=(8, 8))
sc.settings.verbosity = 1 # errors (0), warnings (1), info (2), hints (3)
plt.rcParams["font.family"] = "Arial"
sns.set_style("white")

pd.options.display.max_rows = 9999

start_time = datetime.now()

def print_with_elapsed_time(message):
    elapsed_time = datetime.now() - start_time
    elapsed_seconds = elapsed_time.total_seconds()
    print(f"[{elapsed_seconds:.2f} seconds] {message}")

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [10]:
pip install numpy==1.26.4

   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
    --------------------------------------- 0.3/15.8 MB ? eta -:--:--
   - -------------------------------------- 0.5/15.8 MB 762.0 kB/s eta 0:00:21
   - -------------------------------------- 0.8/15.8 MB 882.6 kB/s eta 0:00:18
   -- ------------------------------------- 1.0/15.8 MB 968.5 kB/s eta 0:00:16
   --- ------------------------------------ 1.3/15.8 MB 1.0 MB/s eta 0:00:15
   --- ------------------------------------ 1.6/15.8 MB 1.0 MB/s eta 0:00:14
   ---- ----------------------------------- 1.8/15.8 MB 1.1 MB/s eta 0:00:13
   ----- ---------------------------------- 2.1/15.8 MB 1.1 MB/s eta 0:00:13
   ----- ---------------------------------- 2.4/15.8 MB 1.2 MB/s eta 0:00:12
   ------ --------------------------------- 2.6/15.8 MB 1.1 MB/s eta 0:00:12
   ------ --------------------------------- 2.6/15.8 MB 1.1 MB/s eta 0:00:12
   ------- ---

In [11]:
# print(f"geopandas version: {gpd.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"scanpy version: {sc.__version__}")

NameError: name 'pd' is not defined

In [3]:
### Directory where the data is stored

# dir = "/mnt/d/Xenium" #Ubuntu
# dir = 'D:\\Xenium'
# dir = "/media/volume/data/spatial/hugo/data" #Ubuntu
# dir = "/media/volume/data/spatial/hugo/data/k5" #Ubuntu
# dir = '/media/volume/volume_spatial/hugo/data/test'
dir = '/media/volume/volume_spatial/hugo/data'

# dir_notebook = 'D:\\Jupyter_notebook/Xenium_jupyter_notebook'
# dir_notebook = '/mnt/d/Jupyter_notebook/Xenium_jupyter_notebook'
# dir_notebook = '/media/volume/data/spatial/hugo/notebook'
dir_notebook = '/media/volume/volume_spatial/hugo/notebook'


In [4]:
from module.misc import sample_name_import

name_dir = "circa-SD"

samples, samples_ids = sample_name_import(name_dir)

print(len(samples))
print(samples)

12
['circa4-IGM-ZT01', 'circa4-IGM-ZT05', 'circa4-IGM-ZT09', 'circa4-IGM-ZT13', 'circa4-IGM-ZT17', 'circa4-IGM-ZT21', 'SD1-ZT01', 'SD1-ZT05', 'SD1-ZT09', 'SD1-ZT13', 'SD1-ZT17', 'SD1-ZT21']


# Data pre-processing

## Import data from Xenium output

In [ ]:
from module.xenium_preprocessing import import_xenium

adata = import_xenium(dir, dir_notebook, samples, samples_ids, name_dir)

In [ ]:
# If you don't want to use MMC, skip to next section

# Stop here and run MapMyCell with the individual h5ad generated.
# Then put the .csv files obtained in the "Correlation_Mapping" folder, renamed as: {sample}_CorrelationMapping.csv


In [ ]:
from module.xenium_preprocessing import mmc_merge
adata = mmc_merge(adata, dir_notebook, name_dir)

In [ ]:
adata.obs['mmc:subclass_name'].isna().sum()

# Should be 0. If not zero, some cell are not annotated.
# Check the name of the CorrelationMapping files; if there is any missing, etc.


In [ ]:
if not os.path.exists(f"h5ad/{name_dir}/"):
   os.makedirs(f"h5ad/{name_dir}/")
adata.write(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_MMC.h5ad.gz", compression='gzip')

In [ ]:
adata = sc.read_h5ad(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_MMC.h5ad.gz")

## Compute quality metrics

In [ ]:
sc.pp.calculate_qc_metrics(adata,  percent_top=(10, 20, 50, 150), inplace=True)

In [ ]:
cprobes = (adata.obs["control_probe_counts"].sum() / adata.obs["total_counts"].sum() * 100)
cwords = (adata.obs["control_codeword_counts"].sum() / adata.obs["total_counts"].sum() * 100)
print(f"Negative DNA probe count % : {cprobes}")
print(f"Negative decoding count % : {cwords}")

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(15, 3))

axs[0].set_title("Total transcripts per cell")
sns.histplot(adata.obs["total_counts"],kde=True,ax=axs[0])

axs[1].set_title("Unique transcripts per cell")
sns.histplot(adata.obs["n_genes_by_counts"],kde=True,ax=axs[1])

axs[2].set_title("Area of segmented cells")
sns.histplot(adata.obs["cell_area"], kde=True, ax=axs[2])

axs[3].set_title("Nucleus ratio")
sns.histplot(adata.obs["nucleus_area"] / adata.obs["cell_area"], kde=True,ax=axs[3])

if not os.path.exists(f"{dir_notebook}/plot/{name_dir}/"):
   os.makedirs(f"{dir_notebook}/plot/{name_dir}/")
plt.savefig(f"{dir_notebook}/plot/{name_dir}/{name_dir}_quality-metrics.svg")

## Normalize, PCA, UMAP

In [ ]:
### Normalize, log1p, scale, PCA, and UMAP
start_time = datetime.now()
print_with_elapsed_time(f"Start")
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, inplace=True)
print_with_elapsed_time(f"Normalize done")
sc.pp.log1p(adata)
print_with_elapsed_time(f"log1p done")

In [5]:
# if not os.path.exists(f"{dir_notebook}/h5ad/{name_dir}/"):
#    os.makedirs(f"{dir_notebook}/h5ad/{name_dir}/")
# adata.write(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_MMC_norm.h5ad.gz", compression='gzip')

adata = sc.read_h5ad(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_MMC_norm.h5ad.gz")

In [6]:
import scanpy.external as sce

start_time = datetime.now()
print_with_elapsed_time(f"Start")
sc.pp.pca(adata)
print_with_elapsed_time(f"PCA done")

[0.00 seconds] Start
[336.46 seconds] PCA done


In [7]:
if not os.path.exists(f"{dir_notebook}/h5ad/{name_dir}/"):
   os.makedirs(f"{dir_notebook}/h5ad/{name_dir}/")
adata.write(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_MMC_norm.h5ad.gz", compression='gzip')

# adata = sc.read_h5ad(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_MMC_norm.h5ad.gz")

In [8]:
adata.obs['run'] = adata.obs_names.map(lambda name: name.split('-')[0]) ### If multiple runs combined in one dataset. Adapat separation sign to the actual name

if adata.obs['run'].nunique() > 1:
    sce.pp.harmony_integrate(adata, key = 'run', basis = f"X_pca",  adjusted_basis = f"X_pca") ### Will overwrite original pca. Change "adjusted_basis" name to save both.
    print_with_elapsed_time(f"Harmony done")

2025-06-11 20:13:25,109 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2025-06-11 20:22:55,553 - harmonypy - INFO - sklearn.KMeans initialization complete.
2025-06-11 20:23:00,769 - harmonypy - INFO - Iteration 1 of 10
2025-06-11 20:30:43,056 - harmonypy - INFO - Iteration 2 of 10
2025-06-11 20:38:17,690 - harmonypy - INFO - Iteration 3 of 10
2025-06-11 20:45:35,542 - harmonypy - INFO - Iteration 4 of 10
2025-06-11 20:53:38,135 - harmonypy - INFO - Iteration 5 of 10
2025-06-11 21:01:22,383 - harmonypy - INFO - Iteration 6 of 10
2025-06-11 21:09:31,444 - harmonypy - INFO - Iteration 7 of 10
2025-06-11 21:12:12,395 - harmonypy - INFO - Converged after 7 iterations


[4114.38 seconds] Harmony done


In [9]:
if not os.path.exists(f"{dir_notebook}/h5ad/{name_dir}/"):
   os.makedirs(f"{dir_notebook}/h5ad/{name_dir}/")
adata.write(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_MMC_norm.h5ad.gz", compression='gzip')

# adata = sc.read_h5ad(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_MMC_norm.h5ad.gz")

In [10]:
sc.pp.neighbors(adata)
print_with_elapsed_time(f"Neighbors done")
sc.tl.umap(adata, min_dist = 1)
print_with_elapsed_time(f"UMAP done")

[4639.44 seconds] Neighbors done
[6473.28 seconds] UMAP done


In [5]:
# if not os.path.exists(f"{dir_notebook}/h5ad/{name_dir}/"):
#    os.makedirs(f"{dir_notebook}/h5ad/{name_dir}/")
# adata.write(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_MMC_norm.h5ad.gz", compression='gzip')

adata = sc.read_h5ad(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_MMC_norm.h5ad.gz")

In [6]:
sc.tl.leiden(adata, resolution = 1) ### Use a higher resolution value to obtain more clusters. They can be adjusted/merged/subclustered later

In [7]:
if not os.path.exists(f"{dir_notebook}/h5ad/{name_dir}/"):
   os.makedirs(f"{dir_notebook}/h5ad/{name_dir}/")
adata.write(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_clusters.h5ad.gz", compression='gzip')

In [ ]:
adata = sc.read_h5ad(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_clusters.h5ad.gz")

## Create a normalised datamatrix

In [8]:
# Create a normalised datamatrix for saving to disk as a csv file - rows are cells, columns are genes
df = pd.DataFrame(data=adata.X.toarray(), index=adata.obs_names, columns=adata.var_names)
df.shape

from module.xenium_preprocessing import add_annotations

df = add_annotations(adata, df)

### Extract normalized expression and clusters for individual cells
if not os.path.exists(f"{dir_notebook}/csv/{name_dir}/"):
   os.makedirs(f"{dir_notebook}/csv/{name_dir}/")
# df.to_csv(f"{dir_notebook}/csv/{name_dir}/{name_dir}_normalized_counts.csv.gz",
#          compression={'method': 'gzip'})
df.to_parquet(f"{dir_notebook}/csv/{name_dir}/{name_dir}_normalized_counts.parquet")
# adata.obs.to_csv(f"{dir_notebook}/csv/{name_dir}/{name_dir}_MMC_norm.csv")

# End of this notebook

Next step : clusters annotation

[v9B_Xenium_annotation_n_subsclustering](./v9B_Xenium_annotation_n_subsclustering.ipynb)